# Build: public datasets for the wider complement / C4b exploration

Downloads are done outside the notebook (`scripts/download_public_datasets.sh`). This notebook turns the downloaded files into harmonised `.h5ad` objects using the loaders in `scripts/oligoc4b_public.py`.

| Dataset | Accession | Species / modality | Comparison | Why |
| --- | --- | --- | --- | --- |
| Park et al. 2023 AD hippocampus | GSE224398 | mouse scRNA-seq | App^NL-G-F^ vs control, 1/3/6 mo | the dataset in which the C4b⁺ DA-oligodendrocyte state was described |
| Aging snRNA-seq HIP + CP | GSE212576 | mouse snRNA-seq | old vs young | aging counterpart already used in this repo |
| Ximerakis et al. 2019 | GSE129788 | mouse scRNA-seq, whole brain | old vs young | independent aging dataset with author cell types |
| Kaya et al. 2022 | GSE202579 | mouse scRNA-seq, white vs grey matter | WM vs GM at 24 mo (WT, Rag1-KO) | interferon-responsive oligodendrocytes in aging white matter |
| Zhou et al. 2020 | GSE140511 | mouse snRNA-seq, cortex + hippocampus | 5XFAD vs non-Tg (± Trem2-KO) | second AD model |
| Chen et al. 2020 | GSE152506 | mouse Spatial Transcriptomics | App^NL-G-F^ vs WT, 3–18 mo | whole-transcriptome spatial AD data; C4b is a plaque-induced gene |
| Jäkel et al. 2019 | GSE118257 | human snRNA-seq, MS white matter | MS lesions vs control | human MS oligodendrocytes |
| Absinta et al. 2021 | GSE180759 | human snRNA-seq, chronic active MS lesions | lesion edge / core / periplaque vs control | human MS lesion rim biology |
| Leng et al. 2021 | GSE147528 | human snRNA-seq, SFG + EC | Braak 6 vs Braak 0 (Braak 2 intermediate) | human AD across pathological stages |
| Sadick et al. 2022 | GSE167494 | human snRNA-seq, PFC, astrocyte/oligodendrocyte-enriched | AD vs non-symptomatic | human AD oligodendrocytes at depth |
| LPC + cuprizone corpus callosum | GSE293850 | mouse snRNA-seq | LPC / cuprizone vs saline | toxic demyelination without adaptive immunity |
| Serpina3n cKO + cuprizone | GSE319903 | mouse snRNA-seq | cuprizone vs normal diet; Serpina3n cKO | Serpina3n is the top C4b-correlated gene in every dataset |
| Lerma-Martin et al. 2024 (snRNA) | GSE279180 | human snRNA-seq, subcortical MS lesions | chronic active / inactive vs control | third human MS single-nucleus dataset, author-annotated |
| Lerma-Martin et al. 2024 (Visium) | GSE279181 | human Visium | same lesions | human MS spatial, whole transcriptome |
| Senescent-like glia in MS, 2025 | GSE277435 | human Visium | MS vs control | second human MS spatial dataset |
| Schirmer et al. 2019 | UCSC `ms` | human snRNA-seq, MS lesions | lesion types vs control | fourth human MS single-nucleus dataset |

Harmonised `obs` columns: `dataset`, `species`, `modality`, `sample`, `group` (comparison level), `group_ref`, `cell_type_coarse`, `cell_type_original`, and `condition_original` for the human lesion / Braak labels. Coarse cell types come from author annotations when provided, otherwise from marker-based cluster annotation (`annotate_by_markers`).

Environment: `OLIGOC4B_PUBLIC_RAW_DIR` (downloaded files, one folder per accession) and `OLIGOC4B_PUBLIC_PROCESSED_DIR` (output).

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "../../scripts")
import importlib, oligoc4b_public as pub
importlib.reload(pub)
import pandas as pd
print("raw:", pub.RAW_DIR); print("processed:", pub.PROCESSED_DIR)
print("available loaders:", list(pub.LOADERS))

In [ ]:
built = pub.build_all(overwrite=False)
built

In [ ]:
import scanpy as sc
rows = []
for name, path in pub.processed_paths().items():
    a = sc.read_h5ad(path, backed="r")
    o = a.obs
    rows.append({"dataset": name, "n_cells": a.n_obs, "n_genes": a.n_vars, "species": o["species"].iloc[0], "modality": o["modality"].iloc[0],
                 "n_samples": o["sample"].nunique(), "groups": o["group"].value_counts().to_dict(),
                 "oligodendrocytes": int((o["cell_type_coarse"].astype(str) == "Oligodendrocyte").sum()),
                 "microglia": int((o["cell_type_coarse"].astype(str) == "Microglia").sum())})
summary = pd.DataFrame(rows).set_index("dataset")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 80)
display(summary)

In [ ]:
# sanity check of the coarse annotation: composition per dataset, author labels vs coarse types, and canonical markers
import numpy as np, scipy.sparse as sp
for name, path in pub.processed_paths().items():
    a = sc.read_h5ad(path, backed="r")
    ct = a.obs["cell_type_coarse"].astype(str)
    print(f"\n{name}: ", ct.value_counts().to_dict())
    if "cell_type_original" in a.obs and a.obs["cell_type_original"].nunique() < 60:
        cross = pd.crosstab(a.obs["cell_type_original"].astype(str), ct)
        display(cross.loc[cross.sum(axis=1).sort_values(ascending=False).index[:25]])
    markers = {"Oligodendrocyte": "Plp1", "Microglia": "Hexb", "Astrocyte": "Aqp4", "Neuron": "Snap25", "OPC": "Pdgfra"}
    if a.obs["species"].iloc[0] == "human":
        markers = {k: v.upper() for k, v in markers.items()}
    rows = {}
    for k, g in markers.items():
        if g not in a.var_names:
            continue
        x = a[:, g].to_memory().X
        x = np.asarray(x.todense()).ravel() if sp.issparse(x) else np.asarray(x).ravel()
        rows[g] = {c: round(float((x[ct.values == c] > 0).mean()), 2) for c in ["Oligodendrocyte", "OPC", "Microglia", "Astrocyte", "Neuron"] if (ct.values == c).sum() > 0}
    if rows:
        print("fraction of cells detecting each marker, by coarse type:"); display(pd.DataFrame(rows).T)